# Build the Movie, Sales and Movie_Sales tables
Outputs three load-ready CSVs: `movie.csv`, `sales_table.csv`, `movie_sales.csv`.

In [ ]:
import pandas as pd

In [ ]:
# ---------- MOVIE TABLE ----------
mv = pd.read_csv("movie_pk.csv")
movie = mv.rename(columns={
    "MovieID": "movieid", "Title": "title", "Studio": "studio", "Runtime": "runtime",
    "Rating": "rating", "ReleaseDate": "reldate", "ProductionBudget": "prodbudget"
})[["movieid", "title", "studio", "runtime", "rating", "reldate", "prodbudget"]].copy()
movie["runtime"] = movie["runtime"].astype("Int64")
movie["prodbudget"] = movie["prodbudget"].astype("Int64")
movie.to_csv("movie.csv", index=False)
movie.info()

In [ ]:
# ---------- MATCH sales rows to movies on title + release year ----------
# sales.csv has no movieid, so we join on title + year to attach one.
mv["year"] = pd.to_datetime(mv["ReleaseDate"], errors="coerce").dt.year
sa = pd.read_csv("sales.csv")
merged = mv.merge(sa, left_on=["Title", "year"], right_on=["title", "year"], how="inner")
# keep one sales record per movie
merged = merged.drop_duplicates(subset="MovieID", keep="first").reset_index(drop=True)
print(len(merged), "movies matched to sales")

In [ ]:
# ---------- SALES TABLE ----------
merged["salesid"] = range(1, len(merged) + 1)
sales = merged.rename(columns={
    "theatre_count": "theater_count",
    "international_box_office": "international",
    "domestic_box_office": "domestic",
})[["salesid", "theater_count", "international", "domestic"]].copy()
# integer columns (no trailing .0) so Postgres INTEGER/BIGINT accepts them
for c in ["theater_count", "international", "domestic"]:
    sales[c] = sales[c].astype("Int64")
sales.to_csv("sales_table.csv", index=False)
sales.head()

In [ ]:
# ---------- MOVIE_SALES JUNCTION ----------
movie_sales = merged.rename(columns={"MovieID": "movieid"})[["movieid", "salesid"]].copy()
movie_sales.to_csv("movie_sales.csv", index=False)

# integrity checks
assert movie_sales["salesid"].is_unique
assert movie_sales["movieid"].isin(movie["movieid"]).all()
assert movie_sales["salesid"].isin(sales["salesid"]).all()
print("FK integrity OK:", len(movie_sales), "rows")